# Chapter 10. 정책 그래디언트 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter10_1_policy_gradient.ipynb)

책 본문: [Chapter 10](https://smhanlab.com/book-ml/kor/ml2/chapter10.html)

이 노트북은 10.1절 '정책 그래디언트 정리'의 핵심 수식을 numpy로 그대로
돌려 봅니다: softmax 정책의 one-vs-rest 그래디언트 모양, π-가중합이 0인
성질, 책의 '손으로 한 번' 스텝 업데이트, 그리고 정규(연속) 정책의
평균 미분.

## 0. 폰트·그림 저장 설정

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", kr[0] if kr else "미발견")

한글 폰트: Noto Sans CJK KR


## 1. softmax 정책: 로짓 → 확률

2행동 정책, 책의 예와 같은 \(\theta = [1, 0]\), 상태 \(s = 1\)
(로짓 = \(\theta s = \theta\)). softmax가 합이 1인 확률로 변환한다.

In [2]:
def softmax(z):
    z = z - z.max()
    return np.exp(z) / np.exp(z).sum()

theta = np.array([1.0, 0.0])   # 책 예: theta = [1, 0], s = 1
p = softmax(theta)
print(f"p0 = {p[0]:.4f},  p1 = {p[1]:.4f}")
# 책: p0 ≈ 0.7311, p1 ≈ 0.2689

p0 = 0.7311,  p1 = 0.2689


## 2. one-vs-rest 그래디언트 \(\nabla_\theta \log \pi\theta(a|s)\)

2행동 softmax에서 '고른 행동' 성분은 \((1-p_a)\), '고르지 않은' 성분은
\(-p_k\): 이 모양의 유래가 '확률합 = 1' 제약이다.

In [3]:
s = 1.0
grad0 = np.array([(1 - p[0]) * s, -p[1] * s])    # ∇log pi(0|s)
grad1 = np.array([-p[0] * s, (1 - p[1]) * s])    # ∇log pi(1|s)
print("∇log pi(0|s) =", np.round(grad0, 4))
print("∇log pi(1|s) =", np.round(grad1, 4))

# 유한 차분으로 수치 검증
eps = 1e-6
def log_prob(k, th):
    return np.log(softmax(th)[k])
e0, e1 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
num0 = np.array([
    (log_prob(0, theta + eps*e0) - log_prob(0, theta - eps*e0)) / (2*eps),
    (log_prob(0, theta + eps*e1) - log_prob(0, theta - eps*e1)) / (2*eps),
])
print("유한차분 검증 (행동 0):", np.allclose(grad0, num0, atol=1e-5))

∇log pi(0|s) = [ 0.2689 -0.2689]
∇log pi(1|s) = [-0.7311  0.7311]
유한차분 검증 (행동 0): True


## 3. 그래디언트의 π-가중합 = 0

\[\mathbb{E}_{a \sim \pi} [\nabla_\theta \log \pi_\theta(a|s)] =
\nabla_\theta \sum_a \pi_\theta(a|s) = \nabla_\theta \, 1 = 0\]

이 '0'이 10.2·10.3절의 '베이스라인을 빼도 편향이 생기지 않는다'
증명의 출발점이다.

In [4]:
weighted = p[0] * grad0 + p[1] * grad1
print("p0·grad0 + p1·grad1 =", np.round(weighted, 10))
assert np.allclose(weighted, 0.0, atol=1e-8)
print("→ E_pi[∇_theta log pi] = 0 확인")

p0·grad0 + p1·grad1 = [0. 0.]
→ E_pi[∇_theta log pi] = 0 확인


## 4. 책의 '손으로 한 번' 스텝 업데이트 (\(\theta=[1,0]\), \(\alpha=0.1\))

**케이스 A**: 행동 0이 샘플링되고 결과 \(G = +1\) (좋은 결과).
**케이스 B**: 행동 1이 샘플링되고 결과 \(G = -1\) (나쁜 결과).

In [5]:
alpha = 0.1
# 케이스 A: a = 0, G = +1
theta_A = theta + alpha * 1.0 * grad0
p_A = softmax(theta_A)
print(f"케이스 A (a=0, G=+1): theta' = {np.round(theta_A, 4)},  p0: {p[0]:.4f} → {p_A[0]:.4f}")

# 케이스 B: a = 1, G = -1
theta_B = theta + alpha * (-1.0) * grad1
p_B = softmax(theta_B)
print(f"케이스 B (a=1, G=-1): theta' = {np.round(theta_B, 4)},  p0: {p[0]:.4f} → {p_B[0]:.4f}")
print("→ 케이스 B: p1이 내려가고, 확률합=1 제약 때문에 p0가 '자동으로' 올라간다 (one-vs-rest)")

케이스 A (a=0, G=+1): theta' = [ 1.0269 -0.0269],  p0: 0.7311 → 0.7415
케이스 B (a=1, G=-1): theta' = [ 1.0731 -0.0731],  p0: 0.7311 → 0.7588
→ 케이스 B: p1이 내려가고, 확률합=1 제약 때문에 p0가 '자동으로' 올라간다 (one-vs-rest)


## 5. 2행동 밴딧으로 REINFORCE: '단일 샘플'이 실제로 굴러가는 모습

행동 0 → 보상 \(+1\), 행동 1 → \(-1\)인 1스텝 에피소드. 매 에피소드
샘플을 하나만 쓰는 (\(N=1\)) REINFORCE로 500에피소드 학습. '기댓값은
정확하지만 분산이 크다'는 10.1절의 말을 숫자로 먼저 맛본다.

In [6]:
rng = np.random.default_rng(42)
T = 500
th = np.zeros(2)          # 초기 정책: p0 = p1 = 0.5
p0_hist, reward_hist = [], []
for t in range(T):
    probs = softmax(th)
    a = rng.choice(2, p=probs)
    r = 1.0 if a == 0 else -1.0
    grad = -probs.copy()
    grad[a] = 1.0 - probs[a]     # one-vs-rest
    th = th + alpha * r * grad   # 경사 상승 (최대화)
    p0_hist.append(probs[0])
    reward_hist.append(r)

first50, last50 = np.mean(reward_hist[:50]), np.mean(reward_hist[-50:])
print(f"평균 보상: 첫 50 에피소드 = {first50:+.3f},  마지막 50 에피소드 = {last50:+.3f}")
print(f"최종 p0 = {p0_hist[-1]:.4f}  (초기 0.5에서 1에 수렴하는 방향)")

평균 보상: 첫 50 에피소드 = +0.520,  마지막 50 에피소드 = +1.000
최종 p0 = 0.9944  (초기 0.5에서 1에 수렴하는 방향)


## 6. 정규 정책(연속 행동): \(\nabla_\mu \log \pi = (a - \mu)/\sigma^2\)

\(\log \pi = -\frac{(a-\mu)^2}{2\sigma^2} + c\)를 \(\mu\)로 미분하면
나온다. 책의 확인 문제 3 숫자: \(a = 5, \mu = 3, \sigma^2 = 1\).

In [7]:
mu, sigma2, a, G = 3.0, 1.0, 5.0, 2.0
grad_mu = (a - mu) / sigma2
print(f"a={a}, mu={mu}, sigma^2={sigma2}:  ∇_mu log pi = {grad_mu}")

alpha = 0.05
mu_new = mu + alpha * G * grad_mu
print(f"갱신 (G=+2, 경사 상승): mu {mu:.2f} → {mu_new:.3f}")
print("→ '샘플링한 행동이 예측 평균보다 크고 결과가 좋았다' → 평균이 a 쪽으로 이동")

a=5.0, mu=3.0, sigma^2=1.0:  ∇_mu log pi = 2.0
갱신 (G=+2, 경사 상승): mu 3.00 → 3.200
→ '샘플링한 행동이 예측 평균보다 크고 결과가 좋았다' → 평균이 a 쪽으로 이동


## 7. 그림

In [8]:
IMG = "/home/smhan/book-ml/kor/src/images"
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.plot(p0_hist, lw=1.2, color="#4c8bf5")
ax.set_title("REINFORCE: p0 learning curve (2-action bandit, seed=42)")
ax.set_xlabel("Episode")
ax.set_ylabel("p0")
ax.set_ylim(0.3, 1.0)
ax.grid(alpha=0.3)

ax = axes[1]
labels = ["Initial\nθ=[1, 0]", "Case A\n(a=0, G=+1)", "Case B\n(a=1, G=-1)"]
vals = [p[0], p_A[0], p_B[0]]
bars = ax.bar(labels, vals, color=["#999999", "#4c8bf5", "#f5a64c"], width=0.55)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.4f}", ha="center")
ax.set_title("p0 after one manual step update (α=0.1)")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(IMG + "/ch10_1_policy_gradient_update.svg", bbox_inches="tight")
plt.show()

## 요약

- softmax 정책의 그래디언트는 **one-vs-rest**: 고른 행동 \((1-p_a)\),
  나머진 \(-p_k\) — '확률합 = 1' 제약에서 온다.
- **π-가중합 = 0** — 10.2·10.3절의 '베이스라인을 빼도 편향 없다'의 출발점.
- 손으로 한 스텝: 좋은 결과는 고른 행동의 확률을 올리고, 나쁜 결과는
  그 확률을 내리면서 *여타 행동의 확률이 자동으로* 올라간다.
- 연속(정규) 정책: \(\nabla_\mu \log \pi = (a-\mu)/\sigma^2\) —
  '예측-실제 차이'가 갱신 방향.
- '기댓값은 정확하지만 분산이 크다'는 장력은 10.2절 REINFORCE에서
  숫자로, 10.3절 베이스라인/Actor-Critic에서 해법으로 다시 만난다.